# Introduction

The data modelling follows the workflow methodology proposed as on https://www.mlsysbook.ai/contents/core/workflow/workflow.html and
https://www.geeksforgeeks.org/machine-learning/machine-learning-lifecycle/

Process followed   
-> Data Collection   
-> Data Preprocessing   
-> Data Exploration and Augmentation   
-> Model Selection : EfficientNet & RestNet50  
-> Model Training and Evaluation   
-> Grad-CAM for interpretability





# Dataset Collection   

-Data collection is done using publicly available medical repositories i.e. Kaggle

## Setting up Kaggle

In [ ]:
from google.colab import files
files.upload()   # <- Create your own kaggle account, generating API key and upload the json file

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## Download the Covid19-radiography-database

In [ ]:
!kaggle datasets download -d tawsifurrahman/covid19-radiography-database
!unzip covid19-radiography-database.zip -d covid19_radiography

## Creating training-test split

In [ ]:
import os
import shutil
import numpy as np

# dataset root
dataset_dir = "covid19_radiography/COVID-19_Radiography_Dataset"
output_dir = "COVID19_split"

# get class folders automatically
classes = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]

# create train/val/test directories
for split in ["train", "val", "test"]:
    for cls in classes:
        for sub in ["images", "masks"]:
            os.makedirs(os.path.join(output_dir, split, cls.replace(" ", "_"), sub), exist_ok=True)

for cls in classes:
    images_src = os.path.join(dataset_dir, cls, "images")
    masks_src  = os.path.join(dataset_dir, cls, "masks")

    # assume image & mask filenames match
    images = sorted(os.listdir(images_src))
    masks  = sorted(os.listdir(masks_src))

    # ensure matching counts
    assert len(images) == len(masks), f"Mismatch in {cls}: {len(images)} images vs {len(masks)} masks"

    np.random.seed(42)
    indices = np.arange(len(images))
    np.random.shuffle(indices)

    n_total = len(indices)
    n_train = int(0.7 * n_total)
    n_val   = int(0.15 * n_total)
    n_test  = n_total - n_train - n_val

    train_idx = indices[:n_train]
    val_idx   = indices[n_train:n_train+n_val]
    test_idx  = indices[n_train+n_val:]

    def copy_files(idxs, dst_split):
        for i in idxs:
            img = images[i]
            mask = masks[i]

            # copy image
            shutil.copy(
                os.path.join(images_src, img),
                os.path.join(output_dir, dst_split, cls.replace(" ", "_"), "images", img)
            )
            # copy mask
            shutil.copy(
                os.path.join(masks_src, mask),
                os.path.join(output_dir, dst_split, cls.replace(" ", "_"), "masks", mask)
            )

    copy_files(train_idx, "train")
    copy_files(val_idx, "val")
    copy_files(test_idx, "test")

# Data Preprocessing

- The data set is preprocessed to ensure uniformity and adaptability for AI modelling

## Image preprocessing

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

DATA_DIR = "COVID19_split"
IMG_SIZE = (256, 256)
BATCH   = 32
SEED    = 42

# Train/Val/Test datasets
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    seed=SEED,
    label_mode="categorical"
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
    label_mode="categorical"
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False,
    label_mode="categorical"
)

class_names = train_ds.class_names
num_classes = len(class_names)
class_names

# Data Exploration and Augmentation

- The processed data set is explored for in depth information and augmentation

## Discovering the data set

In [ ]:
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)

## Checking for class Imbalances

In [ ]:
from collections import Counter
counts = {}
for cls in class_names:
    counts[cls] = len(os.listdir(os.path.join(DATA_DIR, "train", cls, "images")))
counts

total = sum(counts.values())
class_weight = {}
for i, cls in enumerate(class_names):
    class_weight[i] = total / (num_classes * counts[cls])
class_weight

## Data Augmentation layer

In [ ]:
from tensorflow.keras import layers
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Convert to 3 channels and scale to [0,1]
preprocess = keras.Sequential([
    layers.Rescaling(1./255),
])

# Model Selection


## Basic CNN

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_basic_cnn(input_shape=IMG_SIZE + (3,), num_classes=4):
    model = models.Sequential()

    # Block 1
    model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2,2)))
    model.add(layers.Dropout(0.25))

    # Classification head
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

# Build model
model = build_basic_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

## ResNet50

In [ ]:
NUM_CLASSES = len(train_ds.class_names)

# Base ResNet50
base_ResNet50 = tf.keras.applications.ResNet50(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_ResNet50.trainable = False  # freeze for now

# Build model
inputs = layers.Input(shape=IMG_SIZE + (3,))
x = tf.keras.applications.resnet.preprocess_input(inputs)
x = base_ResNet50(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_resNet50 = models.Model(inputs, outputs)

# Compile
model_resNet50.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc", multi_label=False)]
)

model_resNet50.summary()

# Model Training and Evaluation

-Callbacks are defined and training is executed with layer unfreezing

## Model training

### Base CNN

In [ ]:
history_basic_cnn = model.fit(
    train_ds,
    epochs=25,
    validation_data=val_ds,
    class_weight=class_weight,
    callbacks= [
    keras.callbacks.ModelCheckpoint("best_baseline.keras", save_best_only=True, monitor="val_auc", mode="max"),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_auc", mode="max"),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor="val_loss")
]
)

### ResNet50 : Frozen Layers

In [ ]:
history_ResNet50 = model_resNet50.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks= [
        keras.callbacks.ModelCheckpoint("best_resNet50.keras", save_best_only=True, monitor="val_auc", mode="max"),
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_auc", mode="max"),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor="val_loss")
    ],
)

### ResNet50 : Unfreeze Top layers

In [ ]:
base_ResNet50.trainable = True
for layer in base_ResNet50.layers[:-40]:
    layer.trainable = False

model_resNet50.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc", multi_label=False)]
)

In [ ]:
history_ft_ResNet50 = model_resNet50.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[
        keras.callbacks.ModelCheckpoint("best_finetuned_restNet50.keras", save_best_only=True, monitor="val_auc", mode="max"),
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True, monitor="val_auc", mode="max"),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, monitor="val_loss")
    ],
)

## Evaluation

### Plotting Accuracy and Loss curves

In [ ]:
# Defining curve function

import matplotlib.pyplot as plt

def plot_history(history):
    acc      = history.history['accuracy']
    val_acc  = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs   = range(1, len(acc) + 1)

    plt.figure(figsize=(14, 5))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b-', label="Training Accuracy")
    plt.plot(epochs, val_acc, 'r-', label="Validation Accuracy")
    plt.title("Training & Validation Accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b-', label="Training Loss")
    plt.plot(epochs, val_loss, 'r-', label="Validation Loss")
    plt.title("Training & Validation Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()

    plt.show()

### Confusion Matrix

In [ ]:
import os

def evaluate_and_report(model, test_ds, class_names, save_path=None, normalize=False):
    import numpy as np, itertools
    from sklearn.metrics import classification_report, confusion_matrix
    import matplotlib.pyplot as plt

    # Evaluate model
    test_loss, test_acc, test_auc = model.evaluate(test_ds)
    print({"test_loss": test_loss, "test_acc": test_acc, "test_auc": test_auc})

    # Predictions
    y_true = np.concatenate([y for _, y in test_ds], axis=0)
    y_true = np.argmax(y_true, axis=1)
    y_prob = model.predict(test_ds)
    y_pred = np.argmax(y_prob, axis=1)

    # Classification report
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    print(report)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    fig = plt.figure(figsize=(6,5))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)

    # Annotate
    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.colorbar()
    plt.tight_layout()

    # Save results if requested
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)  # ✅ ensure directory exists
        report_path = os.path.splitext(save_path)[0] + ".txt"
        with open(report_path, "w") as f:
            f.write("Test Results\n")
            f.write(f"Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}, AUC: {test_auc:.4f}\n\n")
            f.write(report)

        fig.savefig(save_path + "_cm.png")  # save confusion matrix image
        print(f"Results saved to {report_path} and confusion matrix to {save_path}_cm.png")

    plt.show()



#### Basic CNN

In [ ]:
plot_history(history_basic_cnn)

In [ ]:
evaluate_and_report(model, test_ds, class_names, save_path="results/basic_cnn", normalize=True)

#### ResNet50 : Frozen Top layers

In [ ]:
plot_history(history_ResNet50)

#### ResNet50 : Unfreeze Layers

In [ ]:
plot_history(history_ft_ResNet50)

In [ ]:
evaluate_and_report(model_resNet50, test_ds, class_names, save_path="results/ResNet50", normalize=True)

# Grad-CAM for interpretability

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def display_gradcam(img, heatmap, label, alpha=0.45, cmap='jet'):
    """
    Display Grad-CAM results with enhanced clarity using OpenCV for blending.
    """
    # Normalize heatmap to [0, 255]
    heatmap = np.uint8(255 * heatmap)

    # Apply color map
    heatmap_colored = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Ensure input image is in uint8 format and correct size
    if img.dtype != np.uint8:
        img = np.uint8(255 * (img / np.max(img)))

    # Resize heatmap to match input image
    heatmap_colored = cv2.resize(heatmap_colored, (img.shape[1], img.shape[0]))

    # Overlay heatmap on the original image
    overlay = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)

    # Convert BGR → RGB for matplotlib
    overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # Plot results
    plt.figure(figsize=(15,5))

    plt.subplot(1,3,1)
    plt.imshow(img.astype("uint8"))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,3,2)
    plt.imshow(heatmap_colored)
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1,3,3)
    plt.imshow(overlay)
    plt.title(f"Overlay: {label}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
def make_gradcam_heatmap(img_array, backbone_model, conv_layer_name="top_conv", pred_index=None):
    """
    Computes Grad-CAM heatmap with improved normalization.
    """
    last_conv_layer = backbone_model.get_layer(conv_layer_name)

    grad_model = tf.keras.models.Model(
        inputs=backbone_model.input,
        outputs=[last_conv_layer.output, backbone_model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    # Normalize between 0 and 1
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val == 0:
        heatmap = tf.zeros_like(heatmap)
    else:
        heatmap /= max_val

    return heatmap.numpy()


## Defining Grad-CAM function

In [ ]:
import os, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import resnet50, efficientnet

# --------------------------
# Image preprocessing
# --------------------------
def get_img_array(img_path, size=(224,224), model_type="resnet"):
    img = image.load_img(img_path, target_size=size)
    arr = image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)

    if model_type.lower() == "resnet":
        arr = resnet50.preprocess_input(arr)
    elif model_type.lower() == "efficientnet":
        arr = efficientnet.preprocess_input(arr)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return arr

# --------------------------
# Grad-CAM function (multi-image support)
# --------------------------
def gradcam_from_nested_testdir(model, test_dir, class_names=None, img_size=(224,224),
                                conv_layer_name="top_conv", model_type="efficientnet",
                                backbone_layer_name="efficientnetb0", num_images=5):

    # Detect class subfolders
    all_classes = [d for d in sorted(os.listdir(test_dir)) if os.path.isdir(os.path.join(test_dir, d))]

    image_list = []

    if all_classes:
        # Case A: subfolder structure
        print("Detected classes:", all_classes)
        if class_names is None:
            class_names = all_classes

        for cls in all_classes:
            class_path = os.path.join(test_dir, cls)
            if os.path.isdir(os.path.join(class_path, "images")):
                class_path = os.path.join(class_path, "images")

            for f in os.listdir(class_path):
                if f.lower().endswith((".png",".jpg",".jpeg")):
                    image_list.append((cls, os.path.join(class_path, f)))

    else:
        # Case B: flat directory with images
        print("No class subfolders found → treating test_dir as flat image folder")
        class_names = ["unknown"]
        for f in os.listdir(test_dir):
            if f.lower().endswith((".png",".jpg",".jpeg")):
                image_list.append(("unknown", os.path.join(test_dir, f)))

    if not image_list:
        raise ValueError("No images found in test_dir")

    # Select subset of images
    if num_images is None or num_images > len(image_list):
        selected_images = image_list
    else:
        selected_images = random.sample(image_list, num_images)

    print(f"Processing {len(selected_images)} images...")

    for chosen_class, sample_img_path in selected_images:
        print(f"\nClass: {chosen_class} | File: {os.path.basename(sample_img_path)}")

        # Load image
        arr = get_img_array(sample_img_path, size=img_size, model_type=model_type)

        # Predict using outer model
        preds = model.predict(arr, verbose=0)
        pred_idx = np.argmax(preds[0])
        pred_label = class_names[pred_idx] if pred_idx < len(class_names) else f"class_{pred_idx}"
        print(f"Predicted class: {pred_label} (index {pred_idx})")

        # Get backbone
        backbone = model.get_layer(backbone_layer_name)

        # Compute Grad-CAM heatmap
        heatmap = make_gradcam_heatmap(arr, backbone, conv_layer_name=conv_layer_name, pred_index=pred_idx)

        # Plot results
        img = image.load_img(sample_img_path, target_size=img_size)
        img = image.img_to_array(img).astype("uint8")
        display_gradcam(img, heatmap, pred_label)


## ResNet50

In [ ]:
gradcam_from_nested_testdir(
    model=model_resNet50,
    test_dir=os.path.join(DATA_DIR, "test"),
    img_size=(256, 256),  # default for ResNet50
    conv_layer_name="conv5_block3_out",  # last conv layer in ResNet50
    model_type="resnet",
    backbone_layer_name="resnet50",
    num_images=10
)